In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, sys, os, random
import torch
from datasets import Dataset, Value
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from scipy.special import expit

sys.path.append("/Proyecto/Value-disagreement/Python/Utilities")
import Dict_Object #, text_cleansing

### Datasets

In [ ]:
# Loadiong Data
value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all.csv", sep='|')

value_train_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv", delimiter=',', dtype=int)
value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv", delimiter=',', dtype=int)
value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv", delimiter=',', dtype=int)

train_df = value_set.iloc[value_train_set]
val_df = value_set.iloc[value_val_set]
test_df = value_set.iloc[value_test_set]
test_df

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def predict_on_val(model_ckpt, tokenizer_dir, val_hf_tokenized):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_ckpt,
        num_labels=1
    )
    model.resize_token_embeddings(len(tokenizer))

    args = TrainingArguments(
        output_dir=".",
        do_train=False,
        do_eval=False,
        do_predict=True,
        per_device_eval_batch_size=32,
        seed=0,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=args,
        tokenizer=tokenizer
    )

    out = trainer.predict(val_hf_tokenized)

    logits = out.predictions.squeeze()
    probs = expit(logits)

    df = pd.DataFrame({
        "id": val_hf_tokenized["id"],
        "value": val_hf_tokenized["value"],
        "label": np.array(out.label_ids).astype(int),
        "prob": probs.astype(float)
    })

    return df

def cast_dataset_to_hf(df, abs_label=True):
    df = df.copy()
    if abs_label:
        df["label"] = df["label"].apply(lambda x: abs(x))

    dataset_dict = {
        "id": df["uid"].astype(str).tolist(),
        "text": df["scenario"].astype(str).tolist(),
        "orig_label": df["label"].astype(int).tolist(),
        "value": df["value"].astype(str).tolist(),
    }
    return Dataset.from_dict(dataset_dict)


def hf_dataset_tokenize(hf_dataset, tokenizer, soft_target_type="float", input_concat=True, max_length=256):
    def tokenize_batch(batch):
        if input_concat:
            values = [str(v).lower() for v in batch["value"]]
            texts  = batch["text"]
            # mantené el mismo formato que usaste en training si fue así:
            batched_inputs = [f"<{values[i]}> {tokenizer.sep_token} {texts[i]}" for i in range(len(texts))]
        else:
            batched_inputs = batch["text"]

        enc = tokenizer(
            batched_inputs,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

        enc["labels"] = batch["orig_label"]
        enc["value"]  = batch["value"]
        enc["id"]     = batch["id"] 
        return enc

    remove_cols = [c for c in hf_dataset.column_names if c not in ["id", "value"]]
    tokenized = hf_dataset.map(tokenize_batch, batched=True, remove_columns=remove_cols)

    tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    target_type = Value("float32") if soft_target_type == "float" else Value("int64")
    tokenized = tokenized.cast_column("labels", target_type)

    return tokenized

### threshold by model

In [ ]:
threshold_grid = np.linspace(0.05, 0.95, 181)

# construir VAL base una sola vez
val_df = value_set.iloc[value_val_set].copy()
val_hf = cast_dataset_to_hf(val_df, abs_label=True)

thresholds_per_model = {}

for model_name in ["roberta-base", "microsoft_deberta-v3-base"]:

    if model_name == "roberta-base":
        tokenizer_dir = "/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed1"
        checkpoint    = "/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed1/checkpoint-11617"

    elif model_name == "microsoft_deberta-v3-base":
        tokenizer_dir = "/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed0"
        checkpoint    = "/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed0/checkpoint-13069"

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)

    # sanity check: tokens de valores deben existir (si entrenaste con tokenizer guardado)
    assert "<achievement>" in tokenizer.get_vocab(), f"Tokenizer mal cargado: no contiene <achievement> ({model_name})"

    # tokenizar VAL con el tokenizer de ESTE modelo
    tokenized_val = hf_dataset_tokenize(val_hf, tokenizer, soft_target_type="float", input_concat=True)

    # preds directas sobre VAL
    df_val_preds = predict_on_val(
        model_ckpt=checkpoint,
        tokenizer_dir=tokenizer_dir,
        val_hf_tokenized=tokenized_val
    )

    # calibración por valor
    thresholds_per_model[model_name] = {}
    for value in sorted(df_val_preds["value"].unique()):
        df_val = df_val_preds[df_val_preds["value"] == value]

        best_f1 = -1
        best_t = None

        for t in threshold_grid:
            y_pred = (df_val["prob"] >= t).astype(int)
            f1 = f1_score(df_val["label"].astype(int), y_pred, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)

        thresholds_per_model[model_name][value] = best_t

print(json.dumps(thresholds_per_model, indent=2))


In [ ]:
def calibrate_thresholds(df, prob_col="prob", grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)

    out = {}
    for v in sorted(df["value"].unique()):
        d = df[df["value"] == v]
        y = d["label"].astype(int).values
        p = d[prob_col].astype(float).values

        best_f1, best_t = -1.0, float(grid[0])
        for t in grid:
            yhat = (p >= t).astype(int)
            f1 = f1_score(y, yhat, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)

        out[v] = best_t
    return out


def per_value_f1(df, thresholds, prob_col="prob"):
    out = {}
    for v in sorted(df["value"].unique()):
        d = df[df["value"] == v]
        y = d["label"].astype(int).values
        p = d[prob_col].astype(float).values
        t = thresholds[v]
        yhat = (p >= t).astype(int)
        out[v] = float(f1_score(y, yhat, zero_division=0))
    return out


def merge_preds(df_r, df_d):
    # merge 1:1 por id,value (esto evita cualquier desalineación silenciosa)
    a = df_r.copy()
    b = df_d.copy()
    a["id"] = a["id"].astype(str); a["value"] = a["value"].astype(str)
    b["id"] = b["id"].astype(str); b["value"] = b["value"].astype(str)

    m = a.merge(
        b[["id","value","prob","label"]],
        on=["id","value"],
        how="inner",
        suffixes=("_r","_d"),
        validate="1:1",
    )
    # sanity check: labels iguales
    assert (m["label_r"].values == m["label_d"].values).all()
    m = m.rename(columns={"label_r":"label"})
    return m

# Grid
threshold_grid = np.linspace(0.05, 0.95, 181)

val_df = value_set.iloc[value_val_set].copy()
val_hf = cast_dataset_to_hf(val_df, abs_label=True)

# Paths
paths = {
    "roberta": {
        "tokenizer_dir": "/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed1",
        "checkpoint":    "/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed1/checkpoint-11617",
        "pretty_name":   "roberta-base",
    },
    "deberta": {
        "tokenizer_dir": "/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed0",
        "checkpoint":    "/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed0/checkpoint-13069",
        "pretty_name":   "microsoft_deberta-v3-base",
    },
}

# Models Pred
df_val = {}
for key in ["roberta", "deberta"]:
    tok_dir = paths[key]["tokenizer_dir"]
    ckpt    = paths[key]["checkpoint"]

    tokenizer = AutoTokenizer.from_pretrained(tok_dir, use_fast=True)
    assert "<achievement>" in tokenizer.get_vocab(), f"Tokenizer incorrecto (no contiene <achievement>): {tok_dir}"

    tokenized_val = hf_dataset_tokenize(val_hf, tokenizer, soft_target_type="float", input_concat=True)

    df_val[key] = predict_on_val(
        model_ckpt=ckpt,
        tokenizer_dir=tok_dir,
        val_hf_tokenized=tokenized_val
    )

    df_val[key]["id"] = df_val[key]["id"].astype(str)
    df_val[key]["value"] = df_val[key]["value"].astype(str)



# Ensamble
val_merged = merge_preds(df_val["roberta"], df_val["deberta"])
val_merged["prob_ens"] = 0.5 * val_merged["prob_r"] + 0.5 * val_merged["prob_d"]


# thresholds por valor
thr_roberta  = calibrate_thresholds(df_val["roberta"], prob_col="prob", grid=threshold_grid)
thr_deberta  = calibrate_thresholds(df_val["deberta"], prob_col="prob", grid=threshold_grid)
thr_ensemble = calibrate_thresholds(val_merged, prob_col="prob_ens", grid=threshold_grid)

best_thresholds_per_system = {
    paths["roberta"]["pretty_name"]: thr_roberta,
    paths["deberta"]["pretty_name"]: thr_deberta,
    "ensemble_avg": thr_ensemble,
}

f1_roberta  = per_value_f1(df_val["roberta"], thr_roberta, prob_col="prob")
f1_deberta  = per_value_f1(df_val["deberta"], thr_deberta, prob_col="prob")
f1_ensemble = per_value_f1(val_merged, thr_ensemble, prob_col="prob_ens")

# Best Model
best_system_per_value = {}
best_threshold_per_value = {}

all_values = sorted(val_df["value"].astype(str).unique())

for v in all_values:
    candidates = {
        paths["roberta"]["pretty_name"]: f1_roberta.get(v, -1.0),
        paths["deberta"]["pretty_name"]: f1_deberta.get(v, -1.0),
        "ensemble_avg": f1_ensemble.get(v, -1.0),
    }
    winner = max(candidates, key=candidates.get)
    best_system_per_value[v] = winner

    if winner == paths["roberta"]["pretty_name"]:
        best_threshold_per_value[v] = thr_roberta[v]
    elif winner == paths["deberta"]["pretty_name"]:
        best_threshold_per_value[v] = thr_deberta[v]
    else:
        best_threshold_per_value[v] = thr_ensemble[v]

In [ ]:
# Outputs / prints
print("=== Thresholds por sistema (VAL) ===")
print(json.dumps(best_thresholds_per_system, indent=2)[:2000], "...\n")

print("=== Mejor sistema por valor (VAL) ===")
print(json.dumps(best_system_per_value, indent=2))

print("\n=== Threshold final por valor (del sistema ganador) ===")
print(json.dumps(best_threshold_per_value, indent=2))

In [ ]:
print("=== Thresholds por sistema (VAL) ===")
print(json.dumps(best_thresholds_per_system, indent=2)[:2000], "...\n")

print("=== Mejor sistema por valor (VAL) ===")
print(json.dumps(best_system_per_value, indent=2))

print("\n=== Threshold final por valor (del sistema ganador) ===")
print(json.dumps(best_threshold_per_value, indent=2))

out_dir = "/Proyecto/Value-disagreement/Python/Models"
os.makedirs(out_dir, exist_ok=True)

with open(os.path.join(out_dir, "best_thresholds_per_model.json"), "w") as f:
    json.dump(best_thresholds_per_system, f, indent=2)

with open(os.path.join(out_dir, "best_model_per_value.json"), "w") as f:
    json.dump(best_system_per_value, f, indent=2)

with open(os.path.join(out_dir, "best_threshold_per_value.json"), "w") as f:
    json.dump(best_threshold_per_value, f, indent=2)

print(f"\n Guardado en: {out_dir}")